<a href="https://colab.research.google.com/github/kamrynphipps/Sephora-Data-Strategy-Analysis/blob/main/CIS_450_Individual_Project_Part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, udf, size, explode
from pyspark.sql.types import ArrayType, StringType
from flashtext import KeywordProcessor

In [ ]:
spark = SparkSession.builder.appName("SephoraCrossProductAnalysis").getOrCreate()

In [ ]:
bucket = "kphipps5-cis450-individual-project"
base_path = f"gs://{bucket}/sephora-data/"

review_files = ["reviews_0-250.csv", "reviews_250-500.csv", "reviews_500-750.csv", "reviews_750-1250.csv", "reviews_1250-end.csv"]

product_info_file = "product_info.csv"

review_paths = [base_path + f for f in review_files]
product_path = base_path + product_info_file

In [ ]:
spark_df_reviews = spark.read.csv(
    review_paths,
    header=True,
    inferSchema=True
)

spark_df_products = spark.read.csv(
    product_path,
    header=True,
    inferSchema=True
)

In [ ]:
df = spark_df_reviews.join(spark_df_products, on="product_id", how="left")

In [ ]:
product_list = [row["product_name"].lower()
    for row in spark_df_products.select("product_name").distinct().collect()
    if row["product_name"] is not None
]

In [ ]:
keyword_processor = KeywordProcessor(case_sensitive=False)
keyword_processor.add_keywords_from_list(product_list)

In [ ]:
def extract_products(text):
    if text is None:
        return []
    return keyword_processor.extract_keywords(str(text).lower())

extract_udf = udf(extract_products, ArrayType(StringType()))

In [ ]:
df = df.withColumn(
    "mentioned_products",
    extract_udf(col("review_text"))
)

In [ ]:
df = df.select("product_id", "review_text", "mentioned_products")

In [ ]:
df_exploded = df.withColumn("mentioned_product", explode(col("mentioned_products")))

In [ ]:
co_mentions = df_exploded.groupBy("product_id", "mentioned_product").count()

In [ ]:
co_mentions.show(20, truncate=False)